# Probing LittleLearner's knowledge boundary

**Paper:** *LittleLearner* (arXiv 2608.13545). **Claim:** a model trained only on K-5 text knows K-5 material but not what comes after Grade 5.

**The test, in three steps:**
1. Take two models of the same size, trained the same way: **LittleLearner** (K-5-filtered data) and **Unfiltered** (normal data).
2. Give both a multiple-choice quiz where every question is tagged with a grade (K-8).
3. If the claim holds, LittleLearner's accuracy should drop **much more** than Unfiltered's after Grade 5.

The key number is the **difference-in-differences (DiD)** = (LittleLearner's drop) − (Unfiltered's drop). Both models get worse on harder grades; only an *extra* drop for LittleLearner supports the claim.

**To run:** pick `SIZE`, run all cells. Scores are cached in `cache/`, so re-runs are instant.

In [ ]:
# !pip install -U transformers accelerate pandas matplotlib   # on Colab
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

SIZE = "0.6b"   # "0.6b" (laptop) -> "1.3b" (Colab T4) -> "5b" (A100/L4)
MODELS = {
    "LittleLearner": f"littlelearner/littlelearner-{SIZE}-base",
    "Unfiltered":    f"littlelearner/unfiltered-{SIZE}-base",
}
ITEMS_PATH = Path("data/items.jsonl")

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32
FIG_DIR, CACHE_DIR = Path("figs"), Path("cache")
FIG_DIR.mkdir(exist_ok=True); CACHE_DIR.mkdir(exist_ok=True)

# Deck colours: teal = LittleLearner, slate = Unfiltered, light orange = Beyond-K-5
TEAL, SLATE, ORANGE, ORANGE_BAND, INK = "#00939C", "#5B6B78", "#D9622B", "#FBEFE8", "#33414D"
MODEL_COLORS = {"LittleLearner": TEAL, "Unfiltered": SLATE}
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 200, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#D9E1E6", "axes.axisbelow": True, "font.size": 10,
})
print(DEVICE, DTYPE)

## 1. The quiz

One question per line in `data/items.jsonl`:

```json
{"id": "sci-g7-012", "grade": 7, "subject": "science", "topic": "cells",
 "context": "Question: Which organelle releases energy from food?\nAnswer:",
 "options": [" mitochondria", " nucleus", " cell wall", " vacuole"], "answer": 0, "source": "..."}
```

`grade` 0 = K. `answer` is the index of the correct option. Grades ≤ 5 are band **K5**, the rest **BeyondK5**.

In [ ]:
def load_items(path):
    df = pd.read_json(path, lines=True)
    for col in ["id", "grade", "context", "options", "answer"]:
        assert col in df.columns, f"missing column {col}"
    assert df.id.is_unique, "duplicate ids"
    for col in ["subject", "topic", "source"]:
        if col not in df.columns:
            df[col] = ""
    df["band"] = np.where(df.grade <= 5, "K5", "BeyondK5")
    return df

items = load_items(ITEMS_PATH)
print(len(items), "items")
items.pivot_table(index="grade", columns="subject", values="id", aggfunc="count", fill_value=0)

## 2. Let the models answer

A base model can't be asked a question, so we measure **how likely it finds each option** after the question and take the most likely one as its answer.

Likelihood is divided by the option's length in bytes, so short options aren't favoured and the two models (different tokenizers) stay comparable. `correct_sum` (no length correction) is kept as a robustness check.

We also score every option with **no question** (context = `"Answer:"`). That *baseline* should be at chance; if not, the options give the answer away.

In [ ]:
@torch.no_grad()
def option_logprobs(tok, model, context, options):
    # Summed log-prob of each option given the context.
    ctx_ids = tok(context, add_special_tokens=False)["input_ids"]
    if tok.bos_token_id is not None:
        ctx_ids = [tok.bos_token_id] + ctx_ids
    scores = []
    for opt in options:
        opt_ids = tok(opt, add_special_tokens=False)["input_ids"]
        ids = torch.tensor([ctx_ids + opt_ids], device=DEVICE)
        logp = torch.log_softmax(model(ids).logits[0, :-1].float(), dim=-1)
        target = ids[0, 1:]
        scores.append(logp[torch.arange(len(target)), target][-len(opt_ids):].sum().item())
    return scores

def score_model(name, items):
    # Scores with the question and without it ("-optonly"). Cached; the model is only loaded if something is missing.
    runs = {name: items.context.tolist(), f"{name}-optonly": ["Answer:"] * len(items)}
    model = None
    out = {}
    for tag, contexts in runs.items():
        path = CACHE_DIR / f"{tag}_{SIZE}.jsonl"
        done = {r["id"]: r["logprobs"] for r in map(json.loads, path.read_text().splitlines())} if path.exists() else {}
        with open(path, "a") as f:
            for i, (id_, ctx, opts) in enumerate(zip(items.id, contexts, items.options)):
                if id_ in done:
                    continue
                if model is None:
                    print(f"loading {MODELS[name]}")
                    tok = AutoTokenizer.from_pretrained(MODELS[name])
                    model = AutoModelForCausalLM.from_pretrained(MODELS[name], dtype=DTYPE).to(DEVICE).eval()
                done[id_] = option_logprobs(tok, model, ctx, opts)
                f.write(json.dumps({"id": id_, "model": tag, "logprobs": done[id_]}) + "\n")
                if (i + 1) % 25 == 0:
                    print(f"{tag}: {i + 1}/{len(items)}")
        out[tag] = pd.DataFrame({"id": items.id, "model": name, "logprobs": items.id.map(done)})
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return out[name], out[f"{name}-optonly"]

def add_predictions(items, scores):
    df = scores.merge(items, on="id")
    rows = []
    for lp, opts, a in zip(df.logprobs, df.options, df.answer):
        lp = np.array(lp)
        per_byte = lp / np.array([max(len(o.encode()), 1) for o in opts])
        rows.append({
            "correct": per_byte.argmax() == a,
            "correct_sum": lp.argmax() == a,
            "margin": per_byte[a] - np.delete(per_byte, a).max(),   # >0 = prefers the right answer
        })
    df = pd.concat([df, pd.DataFrame(rows, index=df.index)], axis=1)
    df["chance"] = 1 / df.options.map(len)
    return df

with_q, no_q = zip(*(score_model(name, items) for name in MODELS))
res = add_predictions(items, pd.concat(with_q, ignore_index=True))
base = add_predictions(items, pd.concat(no_q, ignore_index=True))

## 3. Sanity checks (before trusting anything)

1. **Baseline:** accuracy without the question should be close to chance.
2. **Read 10 random items** and check that the decisions make sense.

In [ ]:
display(pd.DataFrame({
    "with question": res.groupby(["model", "band"]).correct.mean(),
    "no question":   base.groupby(["model", "band"]).correct.mean(),
    "chance":        res.groupby(["model", "band"]).chance.mean(),
}).round(3))

both = base.pivot(index="id", columns="model", values="correct").all(axis=1)
print(f"{both.sum()} items answered correctly by BOTH models without the question - check their options:")
display(items[items.id.isin(both[both].index)][["id", "grade", "context", "options", "answer"]])

pd.set_option("display.max_colwidth", 120)
sample = res[res.id.isin(items.id.sample(10, random_state=0))]
sample.pivot_table(index=["id", "grade", "context"], columns="model", values=["correct", "margin"]).round(2)

## 4. Results

**Accuracy per band and the drop**, overall and per subject. Then the **DiD with a 95% bootstrap interval**: resample questions within each band, recompute the DiD 10,000 times. If the interval excludes 0, the extra drop is unlikely to be chance.

In [ ]:
def drop_table(df):
    t = df.pivot_table(index="model", columns="band", values="correct")[["K5", "BeyondK5"]]
    t["drop"] = t.K5 - t.BeyondK5
    return t

def did_bootstrap(df, n_boot=10_000, seed=0):
    wide = df.pivot_table(index=["id", "band"], columns="model", values="correct").reset_index()
    ll, uf = wide.LittleLearner.to_numpy(float), wide.Unfiltered.to_numpy(float)
    k5, bk5 = np.flatnonzero(wide.band == "K5"), np.flatnonzero(wide.band == "BeyondK5")
    did = lambda a, b: (ll[a].mean() - ll[b].mean()) - (uf[a].mean() - uf[b].mean())
    rng = np.random.default_rng(seed)
    boots = [did(rng.choice(k5, len(k5)), rng.choice(bk5, len(bk5))) for _ in range(n_boot)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return {"DiD": did(k5, bk5), "ci_low": lo, "ci_high": hi, "share_boots_<=0": np.mean(np.array(boots) <= 0)}

display(drop_table(res).round(3))
display(pd.concat({s: drop_table(g) for s, g in res.groupby("subject")}).round(3))
pd.DataFrame({"all": did_bootstrap(res), **{s: did_bootstrap(g) for s, g in res.groupby("subject")}}).T.round(3)

In [ ]:
def wilson_ci(k, n, z=1.96):
    p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centre - half, centre + half

fig, ax = plt.subplots(figsize=(6.4, 3.6))
grades = sorted(res.grade.unique())
ax.axvspan(5.5, grades[-1] + 0.5, color=ORANGE_BAND, zorder=0)
ax.text(5.6, 1.02, "Beyond-K-5", color=ORANGE, fontweight="bold", va="bottom")
ax.axhline(res.chance.mean(), color=SLATE, ls=":", lw=1.2)
ax.text(grades[0] - 0.4, res.chance.mean() + 0.02, "chance", color=SLATE, fontsize=8)
for m, g in res.groupby("model"):
    g = g.groupby("grade").correct.agg(["sum", "count"])
    acc = g["sum"] / g["count"]
    lo, hi = np.array([wilson_ci(k, n) for k, n in zip(g["sum"], g["count"])]).T
    ax.errorbar(g.index + (-0.08 if m == "LittleLearner" else 0.08), acc, yerr=[acc - lo, hi - acc],
                color=MODEL_COLORS[m], marker="o", ms=5, lw=2, capsize=2, label=m)
ax.set_xticks(grades, ["K" if g == 0 else str(g) for g in grades])
ax.set_ylim(0, 1.08)
ax.set_xlabel("Grade of item"); ax.set_ylabel("Accuracy (95% CI)")
ax.legend(frameon=False, loc="lower left")
ax.set_title(f"Accuracy by grade, {SIZE} base models", loc="left")
fig.savefig(FIG_DIR / f"grade_curve_{SIZE}.png"); plt.show()

In [ ]:
# How confident are the models? margin > 0 = prefers the right answer.
fig, axes = plt.subplots(1, 2, figsize=(7, 3), sharey=True)
for ax, band in zip(axes, ["K5", "BeyondK5"]):
    for m in MODELS:
        ax.hist(res[(res.model == m) & (res.band == band)].margin, bins=20, alpha=0.6, color=MODEL_COLORS[m], label=m)
    ax.axvline(0, color=INK, lw=1)
    ax.set_title(f"{band} items", loc="left"); ax.set_xlabel("Per-byte log-prob margin")
axes[0].set_ylabel("Items"); axes[0].legend(frameon=False)
fig.savefig(FIG_DIR / f"margins_{SIZE}.png"); plt.show()

## 5. What does LittleLearner know beyond Grade 5?

Beyond-K-5 questions it gets right **with a clear margin**. For each one, ask: is it guessable from K-5 knowledge or common words, is it really taught after Grade 5, or did the content slip through the filter?

In [ ]:
LEAK_MARGIN = 0.05   # TODO: justify the threshold
ll = res[res.model == "LittleLearner"]
leaks = ll[(ll.band == "BeyondK5") & ll.correct & (ll.margin > LEAK_MARGIN)]
print(f"{len(leaks)} confident Beyond-K-5 hits out of {(ll.band == 'BeyondK5').sum()}")
leaks.sort_values("margin", ascending=False)[["id", "grade", "topic", "context", "options", "answer", "margin"]]

## 6. Conclusions (TODO)

To compare model sizes, change `SIZE` and run everything again; each size gets its own cache and figures.

1. **Is there an extra drop for LittleLearner after Grade 5? DiD and CI?**
   …
2. **Sharp at Grade 5, or gradual?**
   …
3. **Does it differ by subject?**
   …
4. **Which Beyond-K-5 questions does it still get right, and why?**
   …
5. **Limitations** (number of items, how grades were assigned, multiple choice ≠ knowledge, model size).
   …
6. **One-sentence verdict.**
   …